We load the package and function to analyse the results

In [ ]:
%load_ext autoreload
%autoreload 2
import pandas as pd
from pathlib import Path
import sys
sys.path.append("..")
from config import methods, windows, rebalance_freq, kinds, rf

Load the result calculated in main.py

In [ ]:
BASE_DIR = Path().resolve().parent
cartella_weights = BASE_DIR / "results" / "weights"
cartella_returns = BASE_DIR / "results" / "returns"
cartella_turnover = BASE_DIR / "results" / "turnover"

w_gmv = {}
w_rp = {}
w_eq = {}
net_gmv = {}
net_rp = {}
r_eq = {}
turnover_gmv = {}
turnover_rp = {}

for m in methods:
    w_gmv[m] = {}
    w_rp[m]  = {}
    net_gmv[m] = {}
    net_rp[m] = {}
    turnover_gmv[m] = {}
    turnover_rp[m] = {}
    for w in windows:
        w_gmv[m][w] = {}
        w_rp[m][w]  = {}
        net_gmv[m][w] = {}
        net_rp[m][w] = {}
        turnover_gmv[m][w] = {}
        turnover_rp[m][w] = {}
        for f in rebalance_freq:
            w_gmv[m][w][f] = pd.read_parquet(cartella_weights / f"gmv_{m}_w{w}_{f}.parquet")
            w_rp[m][w][f]  = pd.read_parquet(cartella_weights / f"rp_{m}_w{w}_{f}.parquet")
            net_gmv[m][w][f] = pd.read_parquet(cartella_returns / f"gmv_{m}_w{w}_{f}.parquet")
            net_rp[m][w][f] = pd.read_parquet(cartella_returns / f"rp_{m}_w{w}_{f}.parquet")
            turnover_gmv[m][w][f] = pd.read_parquet(cartella_turnover / f"gmv_{m}_w{w}_{f}.parquet")
            turnover_rp[m][w][f] = pd.read_parquet(cartella_turnover / f"rp_{m}_w{w}_{f}.parquet")

w_eq = {}
r_eq = {}

for w in windows:
    w_eq[w] = {}
    r_eq[w] = {}
    for f in rebalance_freq:
        w_eq[w][f] = pd.read_parquet(cartella_weights / f"eq_w{w}_{f}.parquet")
        r_eq[w][f] = pd.read_parquet(cartella_returns / f"eq_w{w}_{f}.parquet")

Compute the excel file for further or more easy analysis with the stats for every model and any combinations

In [ ]:
from src.metrics import stats
cartella_tables = BASE_DIR / "results" / "tables"
metrics_gmv = stats(net_gmv, turnover_gmv, methods, windows, rebalance_freq, kinds[0], rf)
metrics_rp = stats(net_rp, turnover_rp, methods, windows, rebalance_freq, kinds[1], rf)
metrics_eq = stats(r_eq, None, None, windows, rebalance_freq, kinds[2], rf)
metrics = pd.concat([metrics_gmv, metrics_rp, metrics_eq], ignore_index = True)
metrics = metrics.sort_values(["strategy", "method", "window", "freq"])
metrics.to_excel(cartella_tables / f'Statistiche.xlsx')

Compute various metrics for analysing in depth the results, and create a pivot table as the principal summary

In [ ]:
pivot= metrics.pivot_table(
    index = ["strategy", "method"],
    columns = ["window", "freq"],
    values = ["sharpe", "return", "drawdown", "turnover"]
)
window_stats = metrics[metrics["strategy"] == "GMV"].groupby("window").agg({
    "sharpe": "mean",
    "volatility": "mean",
    "return": "mean",
    "turnover": "mean"
})
freq_stats = metrics[(metrics["strategy"] == "GMV") & (metrics["window"] == 120)].groupby("freq").agg({
    "sharpe": "mean",
    "volatility": "mean",
    "return": "mean",
    "turnover": "mean"
})
shrink_stats = metrics[(metrics["strategy"] == "GMV") & (metrics["window"] == 120) & (metrics["freq"] == "D")].groupby("method").agg({
    "sharpe": "mean",
    "volatility": "mean",
    "return": "mean",
    "turnover": "mean"
})
comparison = metrics.pivot_table(
    index=["window", "freq"],
    columns = "strategy",
    values = "sharpe"
)
print(pivot)
print(window_stats)
print(freq_stats)
print(shrink_stats)
print(comparison)
pivot.to_excel(cartella_tables / "pivot_sharpe.xlsx")

We can see that the GMV strategy, if we group both method simple and shrink, doesn't perform  well; with a permanent sharpe ratio that decreases with the increment of the window and turnover. In addition, the smallest window has a negative return where the noise is more.

Going more in depth, if we analyse only the simple method and fix a window, we can see an unexpected result; the daily rebalance frequency is the only one with a positive sharpe, but with the greatest volatility 

But, starting from the previous analysis, if we fix the rebalancing frequency and divide the results by method. we can see that the exceptional result is given by shrinks method that reduces the noise while simple method got a negative sharpe ratio

Comparing all sharpe ratios of all models for every window and rebalance strategy, the output is clear. GMV is the worst shrinks method stabilizes the result but doesn't overcome the greatest quantity of noise that makes the model unstable. RP is more robust but the problem of noisy data still remains so it doesn't outperform a basic model such as EQ.

An interesting result is the positive return that all models have with daily rebalancing. This is because daily rebalancing allows the model to capture microtrends that less frequent rebalancing misses. Normally transaction costs erode this advantage, but in this work the fixed cost assumption is likely conservative, so the benefit remains.

Compute some results for getting a clear and immediate visualization of the principal effect of this work

In [ ]:
from src.plot import strategycomparison, windoweffect, shrinkageeffect, drawdownplot
strategycomparison(net_gmv, net_rp, r_eq, "Simple", 120, rebalance_freq)
windoweffect(net_gmv, "Simple", windows, "W")
shrinkageeffect(net_gmv, methods, windows[1], "W")
drawdownplot(net_gmv["Simple"][120]["W"], net_rp["Simple"][120]["W"], r_eq[120]["W"])